In [ ]:
# 🎯 Example 1: Sequential Multi-Agent System (Pipeline)
# 🧠 Scenario

# “A travel company has different employees (agents):

# Planner → decides steps

# Flight Agent → finds flights

# Weather Agent → checks weather

# Decision Agent → gives final answer”

# ================================
# AGENT 1: PLANNER
# ================================
def planner_agent(user_query):
    print("\n[Planner Agent] Creating plan...")
    return ["flight", "weather", "decision"]


# ================================
# AGENT 2: FLIGHT AGENT
# ================================
def flight_agent():
    print("\n[Flight Agent] Fetching flights...")
    return [
        {"airline": "IndiGo", "price": 4500},
        {"airline": "Air India", "price": 5200}
    ]


# ================================
# AGENT 3: WEATHER AGENT
# ================================
def weather_agent():
    print("\n[Weather Agent] Checking weather...")
    return {"condition": "Clear", "temp": 28}


# ================================
# AGENT 4: DECISION AGENT
# ================================
def decision_agent(flights, weather):
    print("\n[Decision Agent] Making decision...")

    cheapest = min(flights, key=lambda x: x["price"])

    if weather["condition"] == "Rain":
        return "Avoid travel due to bad weather"

    return f"Book {cheapest['airline']} at ₹{cheapest['price']}"


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def travel_multi_agent(user_query):
    print("User Query:", user_query)

    plan = planner_agent(user_query)

    flights = None
    weather = None

    for step in plan:
        if step == "flight":
            flights = flight_agent()

        elif step == "weather":
            weather = weather_agent()

        elif step == "decision":
            result = decision_agent(flights, weather)

    return result


# RUN
response = travel_multi_agent("Plan my trip Delhi to Mumbai")
print("\nFinal Answer:", response)

User Query: Plan my trip Delhi to Mumbai

[Planner Agent] Creating plan...

[Flight Agent] Fetching flights...

[Weather Agent] Checking weather...

[Decision Agent] Making decision...

Final Answer: Book IndiGo at ₹4500


In [ ]:
# Scenario
# “A hospital uses different employees (agents) to handle patient care in sequence.”

# ================================
# AGENT 1: Intake Agent (Planner)
# - Collects patient symptoms and history
# - Decides which steps are needed (tests, consultations, etc.)

# ================================
# AGENT 2: Diagnostic Agent
# - Orders lab tests or scans
# - Interprets results and identifies possible conditions

# ================================
# AGENT 3: Treatment Agent
# - Suggests treatment options (medication, therapy, surgery)
# - Considers patient preferences and medical guidelines

# ================================
# AGENT 4: Decision Agent
# - Reviews all inputs (history, diagnostics, treatment options)
# - Provides the final recommendation to the patient
#HOSPITAL MULTI-AGENT SYSTEM

# AGENT 1: INTAKE AGENT (PLANNER)
def intake_agent(patient_query):
    print("\n[Intake Agent] Collecting patient data...")

    # Simulated patient info extraction
    patient_data = {
        "symptoms": ["fever", "cough"],
        "history": ["diabetes"]
    }

    # Plan steps based on symptoms
    plan = ["diagnostic", "treatment", "decision"]

    return patient_data, plan


# ================================
# AGENT 2: DIAGNOSTIC AGENT
# ================================
def diagnostic_agent(patient_data):
    print("\n[Diagnostic Agent] Running diagnostics...")

    # Simulated diagnosis
    diagnostics = {
        "tests": ["blood test", "chest X-ray"],
        "condition": "Flu"
    }

    return diagnostics


# AGENT 3: TREATMENT AGENT
def treatment_agent(patient_data, diagnostics):
    print("\n[Treatment Agent] Suggesting treatments...")

    # Simulated treatment options
    treatments = [
        {"option": "Antiviral Medication", "cost": 1000},
        {"option": "Rest + Hydration", "cost": 200}
    ]

    return treatments


# AGENT 4: DECISION AGENT
def decision_agent(patient_data, diagnostics, treatments):
    print("\n[Decision Agent] Finalizing recommendation...")

    # Choose lowest cost treatment (example logic)
    best_option = min(treatments, key=lambda x: x["cost"])

    if diagnostics["condition"] == "Severe":
        return "Immediate hospitalization required"

    return f"Recommended: {best_option['option']} (Cost: ₹{best_option['cost']})"


# MAIN MULTI-AGENT SYSTEM
def hospital_multi_agent(patient_query):
    print("Patient Query:", patient_query)

    patient_data, plan = intake_agent(patient_query)

    diagnostics = None
    treatments = None

    for step in plan:
        if step == "diagnostic":
            diagnostics = diagnostic_agent(patient_data)

        elif step == "treatment":
            treatments = treatment_agent(patient_data, diagnostics)

        elif step == "decision":
            result = decision_agent(patient_data, diagnostics, treatments)

    return result


# RUN
response = hospital_multi_agent("I have fever and cough")
print("\nFinal Recommendation:", response)

Patient Query: I have fever and cough

[Intake Agent] Collecting patient data...

[Diagnostic Agent] Running diagnostics...

[Treatment Agent] Suggesting treatments...

[Decision Agent] Finalizing recommendation...

Final Recommendation: Recommended: Rest + Hydration (Cost: ₹200)


In [ ]:
pip install requests


In [ ]:
pip install google-generativeai


In [ ]:
from google import genai

# CONFIG (NEW GEMINI SDK)
from google.colab import userdata

API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# AGENT 1: INTAKE AGENT (PLANNER)
def intake_agent(patient_query):
    print("\n[Intake Agent] Collecting patient data...")

    prompt = f"""
    Extract symptoms from this patient query: "{patient_query}".
    Return only a comma-separated list (e.g., fever, cough).
    """

    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt
        )

        symptoms_text = response.text.strip()
        symptoms = [s.strip().lower() for s in symptoms_text.split(",")]

    except Exception as e:
        print("Intake Error:", str(e))
        symptoms = ["fever", "cough"]  # fallback

    patient_data = {
        "symptoms": symptoms,
        "history": ["diabetes"],
        "preferences": {"low_cost": True}
    }

    plan = ["diagnostic", "treatment", "quality", "decision"]

    return patient_data, plan


# AGENT 2: DIAGNOSTIC AGENT
def diagnostic_agent(patient_data):
    print("\n[Diagnostic Agent] Using Gemini API...")

    symptoms = ", ".join(patient_data["symptoms"])

    prompt = f"""
    A patient has the following symptoms: {symptoms}.
    What is the most likely disease? Answer in one word only.
    """

    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt
        )

        condition = response.text.strip()

        return {
            "condition": condition,
            "confidence": 0.8
        }

    except Exception as e:
        print("Diagnostic Error:", str(e))
        return {
            "condition": "Unknown",
            "confidence": 0.3
        }


# AGENT 3: TREATMENT AGENT
def treatment_agent(patient_data, diagnostics):
    print("\n[Treatment Agent] Suggesting treatments...")

    condition = diagnostics["condition"].lower()

    if "flu" in condition or "cold" in condition:
        treatments = [
            {"option": "Rest + Hydration", "cost": 200},
            {"option": "Paracetamol", "cost": 100}
        ]
    else:
        treatments = [
            {"option": "Consult Doctor", "cost": 1000}
        ]

    return treatments


# AGENT 4: QUALITY CHECK AGENT
def quality_check_agent(diagnostics, treatments):
    print("\n[Quality Agent] Evaluating output...")

    issues = []

    if diagnostics["confidence"] < 0.6:
        issues.append("Low confidence diagnosis")

    if diagnostics["condition"].lower() == "unknown":
        issues.append("Unclear diagnosis")

    if not treatments:
        issues.append("No treatment options")

    if issues:
        return {"status": "REVIEW", "issues": issues}

    return {"status": "OK", "issues": []}


# AGENT 5: DECISION AGENT
def decision_agent(patient_data, diagnostics, treatments, quality):
    print("\n[Decision Agent] Finalizing recommendation...")

    if quality["status"] == "REVIEW":
        return f"Needs doctor review: {quality['issues']}"

    best = min(treatments, key=lambda x: x["cost"])

    return (
        f"\nCondition: {diagnostics['condition']}\n"
        f"Confidence: {diagnostics['confidence']}\n"
        f"Recommended: {best['option']} (₹{best['cost']})"
    )


# MAIN MULTI-AGENT SYSTEM
def hospital_multi_agent(patient_query):
    print("Patient Query:", patient_query)

    patient_data, plan = intake_agent(patient_query)

    diagnostics = None
    treatments = None
    quality = None
    result = None

    for step in plan:
        if step == "diagnostic":
            diagnostics = diagnostic_agent(patient_data)

        elif step == "treatment":
            treatments = treatment_agent(patient_data, diagnostics)

        elif step == "quality":
            quality = quality_check_agent(diagnostics, treatments)

        elif step == "decision":
            result = decision_agent(patient_data, diagnostics, treatments, quality)

    return result


# RUN SYSTEM
if __name__ == "__main__":
    response = hospital_multi_agent("I have fever and cough and body pain")
    print("\nFinal Recommendation:", response)

Patient Query: I have fever and cough and body pain

[Intake Agent] Collecting patient data...
Intake Error: name 'client' is not defined

[Diagnostic Agent] Using Gemini API...
Diagnostic Error: name 'client' is not defined

[Treatment Agent] Suggesting treatments...

[Quality Agent] Evaluating output...

[Decision Agent] Finalizing recommendation...

Final Recommendation: Needs doctor review: ['Low confidence diagnosis', 'Unclear diagnosis']


In [ ]:
import requests
import time

# CONFIG (HF FREE API)
from google.colab import userdata

API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# SAFE API CALL (IMPORTANT)
def query_hf(prompt):
    payload = {"inputs": prompt}

    response = requests.post(API_URL, headers=HEADERS, json=payload)

    print("Status:", response.status_code)
    print("Response:", response.text)

    if response.status_code == 200:
        return response.json()[0]["generated_text"]

    return "Unknown"


# AGENT 1: INTAKE AGENT
def intake_agent(patient_query):
    print("\n[Intake Agent] Collecting patient data...")

    prompt = f"""
    Extract symptoms from: {patient_query}.
    Return comma separated list.
    """

    result = query_hf(prompt)

    symptoms = [s.strip().lower() for s in result.split(",")]

    patient_data = {
        "symptoms": symptoms,
        "history": ["diabetes"],
        "preferences": {"low_cost": True}
    }

    plan = ["diagnostic", "treatment", "quality", "decision"]

    return patient_data, plan


# AGENT 2: DIAGNOSTIC AGENT
def diagnostic_agent(patient_data):
    print("\n[Diagnostic Agent] Using Hugging Face...")

    symptoms = ", ".join(patient_data["symptoms"])

    prompt = f"""
    Symptoms: {symptoms}
    What is the disease? Answer one word.
    """

    result = query_hf(prompt)

    return {
        "condition": result.strip(),
        "confidence": 0.7
    }


# AGENT 3: TREATMENT AGENT
def treatment_agent(patient_data, diagnostics):
    print("\n[Treatment Agent] Suggesting treatments...")

    condition = diagnostics["condition"].lower()

    if "flu" in condition or "cold" in condition:
        treatments = [
            {"option": "Rest + Hydration", "cost": 200},
            {"option": "Paracetamol", "cost": 100}
        ]
    else:
        treatments = [
            {"option": "Consult Doctor", "cost": 1000}
        ]

    return treatments


# AGENT 4: QUALITY AGENT
def quality_check_agent(diagnostics, treatments):
    print("\n[Quality Agent] Evaluating output...")

    issues = []

    if diagnostics["condition"].lower() == "unknown":
        issues.append("Unclear diagnosis")

    if not treatments:
        issues.append("No treatment options")

    if issues:
        return {"status": "REVIEW", "issues": issues}

    return {"status": "OK", "issues": []}


# AGENT 5: DECISION AGENT
def decision_agent(patient_data, diagnostics, treatments, quality):
    print("\n[Decision Agent] Finalizing recommendation...")

    if quality["status"] == "REVIEW":
        return f"⚠️ Needs doctor review: {quality['issues']}"

    best = min(treatments, key=lambda x: x["cost"])

    return f"""
Condition: {diagnostics['condition']}
Recommended: {best['option']} (₹{best['cost']})
"""


# MAIN SYSTEM
def hospital_multi_agent(query):
    print("Patient Query:", query)

    patient_data, plan = intake_agent(query)

    diagnostics = None
    treatments = None
    quality = None

    for step in plan:
        if step == "diagnostic":
            diagnostics = diagnostic_agent(patient_data)

        elif step == "treatment":
            treatments = treatment_agent(patient_data, diagnostics)

        elif step == "quality":
            quality = quality_check_agent(diagnostics, treatments)

        elif step == "decision":
            result = decision_agent(patient_data, diagnostics, treatments, quality)

    return result


# RUN
print(hospital_multi_agent("I have fever and cough and body pain"))

Patient Query: I have fever and cough and body pain

[Intake Agent] Collecting patient data...
Status: 400
Response: {"error":{"message":"Input required: specify \"prompt\" or \"messages\"","code":400},"user_id":"user_3B9WMrZMs4nrT4JKNNVEh0RiTIH"}

[Diagnostic Agent] Using Hugging Face...
Status: 400
Response: {"error":{"message":"Input required: specify \"prompt\" or \"messages\"","code":400},"user_id":"user_3B9WMrZMs4nrT4JKNNVEh0RiTIH"}

[Treatment Agent] Suggesting treatments...

[Quality Agent] Evaluating output...

[Decision Agent] Finalizing recommendation...
⚠️ Needs doctor review: ['Unclear diagnosis']


In [ ]:
import requests

# ================================
# CONFIG (OPENROUTER API)
# ================================
from google.colab import userdata

API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# GENERIC API CALL
def call_llm(prompt):
    response = requests.post(
        API_URL,
        headers=HEADERS,
        json={
            "model": "openai/gpt-3.5-turbo",
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print("API Error:", response.text)
        return "Unknown"


# AGENT 1: INTAKE AGENT
def intake_agent(query):
    print("\n[Intake Agent] Extracting symptoms...")

    prompt = f"Extract symptoms from: {query}. Return comma separated."

    result = call_llm(prompt)

    symptoms = [s.strip().lower() for s in result.split(",")]

    patient_data = {
        "symptoms": symptoms,
        "history": ["diabetes"],
        "preferences": {"low_cost": True}
    }

    plan = ["diagnostic", "treatment", "quality", "decision"]

    return patient_data, plan


# AGENT 2: DIAGNOSTIC AGENT (DETAILED)
def diagnostic_agent(data):
    print("\n[Diagnostic Agent] Diagnosing...")

    symptoms = ", ".join(data["symptoms"])

    prompt = f"""
    A patient has these symptoms: {symptoms}.

    Provide:
    1. Disease name
    2. Short explanation
    3. Key symptoms

    Format:
    Disease: ...
    Explanation: ...
    Symptoms: ...
    """

    result = call_llm(prompt)

    return {
        "raw": result,
        "confidence": 0.85
    }


# AGENT 3: TREATMENT AGENT
def treatment_agent(data, diag):
    print("\n[Treatment Agent] Suggesting treatments...")

    # Basic fallback logic
    if "flu" in diag["raw"].lower() or "influenza" in diag["raw"].lower():
        return [
            {"option": "Rest + Hydration", "cost": 200},
            {"option": "Paracetamol", "cost": 100}
        ]

    return [{"option": "Consult Doctor", "cost": 1000}]


# AGENT 4: QUALITY AGENT
def quality_agent(diag, treatments):
    print("\n[Quality Agent] Checking quality...")

    if "unknown" in diag["raw"].lower():
        return {"status": "REVIEW", "issues": ["Unclear diagnosis"]}

    return {"status": "OK", "issues": []}


# AGENT 5: DECISION AGENT (DETAILED OUTPUT)
def decision_agent(data, diag, treatments, quality):
    print("\n[Decision Agent] Finalizing...")

    if quality["status"] == "REVIEW":
        return f"⚠️ Needs doctor review: {quality['issues']}"

    best = min(treatments, key=lambda x: x["cost"])

    prompt = f"""
    Based on this diagnosis:

    {diag['raw']}

    Suggested treatment: {best['option']}

    Generate a detailed patient-friendly report including:

    - Condition
    - Explanation
    - Symptoms
    - Treatment Plan
    - Precautions

    Keep it simple and clear.
    """

    final = call_llm(prompt)

    return final


# MAIN SYSTEM
def run_system(query):
    print("Patient Query:", query)

    data, plan = intake_agent(query)

    for step in plan:
        if step == "diagnostic":
            diag = diagnostic_agent(data)

        elif step == "treatment":
            treat = treatment_agent(data, diag)

        elif step == "quality":
            quality = quality_agent(diag, treat)

        elif step == "decision":
            result = decision_agent(data, diag, treat, quality)

    return result


# RUN
if __name__ == "__main__":
    response = run_system("I have fever and cough and body pain")
    print("\nFinal Recommendation:\n", response)

Patient Query: I have fever and cough and body pain

[Intake Agent] Extracting symptoms...

[Diagnostic Agent] Diagnosing...

[Treatment Agent] Suggesting treatments...

[Quality Agent] Checking quality...

[Decision Agent] Finalizing...

Final Recommendation:
 Dear Patient,

Condition:
You have been diagnosed with Influenza (flu), which is a viral infection that commonly affects the respiratory system.

Explanation:
Influenza spreads easily from person to person, especially in crowded places, and can cause symptoms like fever, cough, body pain, fatigue, sore throat, runny or stuffy nose, headache, chills, and sometimes digestive issues like nausea and diarrhea.

Symptoms:
- Fever
- Cough
- Body pain
- Fatigue
- Sore throat
- Runny or stuffy nose
- Headache
- Chills
- Digestive issues like nausea and diarrhea

Treatment Plan:
The suggested treatment for Influenza is to take Paracetamol to help alleviate your symptoms and reduce fever.

Precautions:
- Get plenty of rest
- Drink plenty o

In [ ]:
# Scenario: Corporate Market Research & Strategy
# A company wants to explore launching a new product in a competitive market. The Manager Agent oversees the process and delegates tasks to specialized workers.

# 👩‍💼 Manager Agent
# - Receives the overall goal: “Evaluate feasibility of launching Product Y in Asia.”
# - Dynamically assigns tasks to worker agents depending on what’s needed.
# - Example: If budget is unclear → send to Finance Worker. If regulations are complex → send to Legal Worker.

# ================================
# 📊 Worker Agents
# ================================
# - Market Research Worker
# - Collects competitor data, customer preferences, and demand forecasts.
# - Reports: “High demand in Tier‑1 cities, moderate competition.”
# - Finance Worker
# - Analyzes budget, ROI, and pricing strategy.
# - Reports: “Estimated $10M investment, ROI in 2 years.”
# - Operations Worker
# - Evaluates supply chain, production capacity, and logistics.
# - Reports: “Factories can scale up, but shipping costs are high.”
# - Legal Worker
# - Reviews compliance, intellectual property, and regional regulations.
# - Reports: “Trademark available, but import laws require certification.”
# - HR Worker
# - Assesses staffing needs and training requirements.
# - Reports: “Need 50 new hires for customer support and sales.”
import requests

# CONFIG (OPENROUTER API)
from google.colab import userdata

API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
# GENERIC API CALL
def call_llm(prompt):
    response = requests.post(
        API_URL,
        headers=HEADERS,
        json={
            "model": "openai/gpt-3.5-turbo",
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print("API Error:", response.text)
        return "Error"


# MANAGER AGENT
def manager_agent(goal):
    print("\n[Manager Agent] Analyzing goal...")

    prompt = f"""
    Goal: {goal}

    Decide which workers are needed from:
    - market
    - finance
    - operations
    - legal
    - hr

    Return comma separated list.
    """

    result = call_llm(prompt)

    workers = [w.strip().lower() for w in result.split(",")]

    print("Assigned Workers:", workers)

    return workers


# WORKER AGENTS

def market_worker(goal):
    print("\n[Market Research Worker] Working...")

    prompt = f"""
    Analyze market for: {goal}

    Provide:
    - Demand
    - Competition
    - Customer trends
    """

    return call_llm(prompt)


def finance_worker(goal):
    print("\n[Finance Worker] Working...")

    prompt = f"""
    Analyze financial feasibility for: {goal}

    Provide:
    - Investment needed
    - ROI timeline
    - Pricing strategy
    """

    return call_llm(prompt)


def operations_worker(goal):
    print("\n[Operations Worker] Working...")

    prompt = f"""
    Evaluate operations for: {goal}

    Provide:
    - Supply chain
    - Production capacity
    - Logistics challenges
    """

    return call_llm(prompt)


def legal_worker(goal):
    print("\n[Legal Worker] Working...")

    prompt = f"""
    Analyze legal aspects for: {goal}

    Provide:
    - Regulations
    - Compliance
    - Risks
    """

    return call_llm(prompt)


def hr_worker(goal):
    print("\n[HR Worker] Working...")

    prompt = f"""
    Analyze HR needs for: {goal}

    Provide:
    - Hiring needs
    - Skills required
    - Training
    """

    return call_llm(prompt)


# DECISION AGENT
def decision_agent(goal, reports):
    print("\n[Decision Agent] Creating final strategy...")

    combined_reports = "\n\n".join(reports)

    prompt = f"""
    Goal: {goal}

    Based on these reports:

    {combined_reports}

    Provide a FINAL STRATEGY including:
    - Feasibility (Yes/No with reason)
    - Key Opportunities
    - Risks
    - Recommended Action Plan
    """

    return call_llm(prompt)


# MAIN SYSTEM
def corporate_multi_agent(goal):
    print("Goal:", goal)

    workers = manager_agent(goal)

    reports = []

    for worker in workers:
        if "market" in worker:
            reports.append(market_worker(goal))

        elif "finance" in worker:
            reports.append(finance_worker(goal))

        elif "operations" in worker:
            reports.append(operations_worker(goal))

        elif "legal" in worker:
            reports.append(legal_worker(goal))

        elif "hr" in worker:
            reports.append(hr_worker(goal))

    final_decision = decision_agent(goal, reports)

    return final_decision


# RUN
if __name__ == "__main__":
    result = corporate_multi_agent(
        "Evaluate feasibility of launching Product Y in Asia"
    )

    print("\nFinal Strategy:\n", result)

Goal: Evaluate feasibility of launching Product Y in Asia

[Manager Agent] Analyzing goal...
Assigned Workers: ['to evaluate the feasibility of launching product y in asia', 'workers from market', 'finance', 'operations', 'legal', 'and hr departments are needed.\n\nworkers needed: market', 'finance', 'operations', 'legal', 'hr']

[Market Research Worker] Working...

[Finance Worker] Working...

[Operations Worker] Working...

[Legal Worker] Working...

[Market Research Worker] Working...

[Finance Worker] Working...

[Operations Worker] Working...

[Legal Worker] Working...

[HR Worker] Working...

[Decision Agent] Creating final strategy...

Final Strategy:
 Final Strategy:

Feasibility: Yes

Reason: Despite the initial investment needed and potential risks, the market research indicates a demand for Product Y in Asia, and the competitive landscape presents opportunities for success. By carefully addressing regulatory compliance, supply chain logistics, and market trends, launching Pr

In [ ]:
# Agent 1: Crisis Coordinator (Broadcaster)
# - Broadcasts: “Data breach detected in customer database. Immediate response required.”
# - Sends this to all other agents simultaneously.

# 🛡️ Agent 2: IT Security Agent
# - Receives broadcast.
# - Responds: “Isolate affected servers, patch vulnerabilities, start forensic analysis.”

# 📞 Agent 3: Communications Agent
# - Receives broadcast.
# - Responds: “Draft internal memo, prepare press release, notify stakeholders.”

# 💰 Agent 4: Finance Agent
# - Receives broadcast.
# - Responds: “Estimate financial impact, allocate emergency funds, review insurance coverage.”

# 👩‍⚖️ Agent 5: Legal Agent
# - Receives broadcast.
# - Responds: “Assess regulatory obligations, prepare compliance reports, advise on liability.”

# 👩‍💼 Agent 6: HR Agent
# - Receives broadcast.
# - Responds: “Brief employees, provide guidance on handling customer queries, ensure morale support.”

# 🧑‍⚖️ Agent 7: Decision Agent (Coordinator)
# - Collects all responses.
# - Integrates into a final crisis response plan:
# “Servers isolated, communications prepared, financial impact assessed, compliance secured, employees briefed.”
import requests

# CONFIG (OPENROUTER API)
from google.colab import userdata

API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
# GENERIC API CALL
def call_llm(prompt):
    response = requests.post(
        API_URL,
        headers=HEADERS,
        json={
            "model": "openai/gpt-3.5-turbo",
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print("API Error:", response.text)
        return "Error"


# AGENT 1: CRISIS COORDINATOR (BROADCAST)
def crisis_coordinator():
    print("\n[Crisis Coordinator] Broadcasting alert...")

    message = "Data breach detected in customer database. Immediate response required."

    return message


# AGENT 2: IT SECURITY
def it_security_agent(message):
    print("\n[IT Security Agent] Responding...")

    prompt = f"""
    Crisis: {message}

    Provide immediate IT security actions.
    """

    return call_llm(prompt)


# AGENT 3: COMMUNICATIONS
def communications_agent(message):
    print("\n[Communications Agent] Responding...")

    prompt = f"""
    Crisis: {message}

    Provide communication strategy for employees, customers, and media.
    """

    return call_llm(prompt)


# AGENT 4: FINANCE
def finance_agent(message):
    print("\n[Finance Agent] Responding...")

    prompt = f"""
    Crisis: {message}

    Provide financial response plan including cost estimation and emergency funding.
    """

    return call_llm(prompt)

# AGENT 5: LEGAL
def legal_agent(message):
    print("\n[Legal Agent] Responding...")

    prompt = f"""
    Crisis: {message}

    Provide legal actions including compliance and liability considerations.
    """

    return call_llm(prompt)


# AGENT 6: HR
def hr_agent(message):
    print("\n[HR Agent] Responding...")

    prompt = f"""
    Crisis: {message}

    Provide HR actions including employee communication and support.
    """

    return call_llm(prompt)


# AGENT 7: DECISION AGENT
def decision_agent(responses):
    print("\n[Decision Agent] Integrating responses...")

    combined = "\n\n".join(responses)

    prompt = f"""
    Based on the following crisis responses:

    {combined}

    Create a FINAL unified crisis response plan including:
    - Immediate actions
    - Communication plan
    - Financial handling
    - Legal compliance
    - Employee management

    Keep it structured and clear.
    """

    return call_llm(prompt)


# MAIN SYSTEM (BROADCAST FLOW)
def crisis_multi_agent_system():
    # Step 1: Broadcast
    message = crisis_coordinator()

    # Step 2: All agents respond (parallel style)
    responses = []

    responses.append(it_security_agent(message))
    responses.append(communications_agent(message))
    responses.append(finance_agent(message))
    responses.append(legal_agent(message))
    responses.append(hr_agent(message))

    # Step 3: Final decision
    final_plan = decision_agent(responses)

    return final_plan


# RUN
if __name__ == "__main__":
    result = crisis_multi_agent_system()
    print("\n🚨 Final Crisis Plan:\n", result)


[Crisis Coordinator] Broadcasting alert...

[IT Security Agent] Responding...

[Communications Agent] Responding...

[Finance Agent] Responding...

[Legal Agent] Responding...

[HR Agent] Responding...

[Decision Agent] Integrating responses...

🚨 Final Crisis Plan:
 Unified Crisis Response Plan:

Immediate Actions:
1. Identify and contain the breach: Determine the extent of the breach, isolate affected systems, and prevent further unauthorized access.
2. Notify affected customers and relevant authorities promptly to comply with data breach notification laws.
3. Conduct a thorough investigation with cybersecurity experts to identify vulnerabilities and remediate them.
4. Enhanced security measures: Implement data loss prevention measures, update security protocols, and monitor for further threats.
5. Engage with stakeholders: Communicate transparently with senior management, IT security team, legal department, and employees.

Communication Plan:
Internal:
- Immediately notify all empl

In [ ]:
import asyncio
import requests

from google.colab import userdata

API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# ASYNC API CALL
async def call_llm(prompt):
    loop = asyncio.get_event_loop()

    def sync_call():
        response = requests.post(
            API_URL,
            headers=HEADERS,
            json={
                "model": "openai/gpt-3.5-turbo",
                "messages": [
                    {"role": "user", "content": prompt}
                ]
            }
        )

        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]
        else:
            return f"Error: {response.text}"

    return await loop.run_in_executor(None, sync_call)


# COORDINATOR (BROADCAST)
def broadcast():
    print("\n[Coordinator] Broadcasting launch plan...")
    return "Product X launch in Q3, target market North America, budget $5M."


# MARKETING AGENT
async def marketing_agent(message):
    print("[Marketing Agent] Working...")

    prompt = f"""
    Launch Details: {message}

    Create marketing strategy including:
    - Campaigns
    - Channels
    - Positioning
    """

    return await call_llm(prompt)


# FINANCE AGENT
async def finance_agent(message):
    print("[Finance Agent] Working...")

    prompt = f"""
    Launch Details: {message}

    Provide:
    - Budget allocation
    - ROI forecast
    """

    return await call_llm(prompt)


# OPERATIONS AGENT
async def operations_agent(message):
    print("[Operations Agent] Working...")

    prompt = f"""
    Launch Details: {message}

    Provide:
    - Production plan
    - Supply chain strategy
    """

    return await call_llm(prompt)


# LEGAL AGENT
async def legal_agent(message):
    print("[Legal Agent] Working...")

    prompt = f"""
    Launch Details: {message}

    Provide:
    - Compliance requirements
    - Contracts and risks
    """

    return await call_llm(prompt)


# HR AGENT
async def hr_agent(message):
    print("[HR Agent] Working...")

    prompt = f"""
    Launch Details: {message}

    Provide:
    - Hiring needs
    - Training plan
    """

    return await call_llm(prompt)


# DECISION AGENT
async def decision_agent(responses, message):
    print("\n[Decision Agent] Creating final launch plan...")

    combined = "\n\n".join(responses)

    prompt = f"""
    Launch Details: {message}

    Department Reports:
    {combined}

    Create a FINAL CORPORATE LAUNCH PLAN including:
    - Market Strategy
    - Financial Plan
    - Operations Plan
    - Legal Compliance
    - HR Plan
    - Final Recommendation

    Make it structured and professional.
    """

    return await call_llm(prompt)


# MAIN SYSTEM (ASYNC BROADCAST)
async def corporate_launch_system():
    message = broadcast()

    # Run all agents in parallel
    responses = await asyncio.gather(
        marketing_agent(message),
        finance_agent(message),
        operations_agent(message),
        legal_agent(message),
        hr_agent(message)
    )

    # Final decision
    final_plan = await decision_agent(responses, message)

    return final_plan


# SAFE RUN (Works in Jupyter + Python both)
async def main():
    result = await corporate_launch_system()
    print("\n🚀 Final Launch Plan:\n", result)


# RUN
if __name__ == "__main__":
    try:
        asyncio.run(main())   # normal python
    except RuntimeError:
        # Jupyter/Colab case
        import nest_asyncio
        nest_asyncio.apply()
        loop = asyncio.get_event_loop()
        loop.run_until_complete(main())


[Coordinator] Broadcasting launch plan...
[Marketing Agent] Working...
[Finance Agent] Working...
[Operations Agent] Working...
[Legal Agent] Working...
[HR Agent] Working...

[Decision Agent] Creating final launch plan...

🚀 Final Launch Plan:
 FINAL CORPORATE LAUNCH PLAN

Market Strategy:
Our market strategy for the launch of Product X in Q3 will focus on targeting the North American market through a comprehensive marketing campaign that includes digital advertising, influencer partnerships, email marketing, and a product launch event. We will emphasize the unique features and value proposition of Product X, tailoring our messaging to resonate with the specific needs and preferences of the target audience. By utilizing social media platforms, online retailers, industry publications, and influencer collaborations, we aim to generate awareness, drive engagement, and ultimately drive sales in the North American market.

Financial Plan:
With a budget of $5M allocated for the launch of P

In [ ]:
pip install requests asyncio


In [ ]:
# IMPORTS
import asyncio
import requests
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata

API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
# LLM CALL FUNCTION
async def call_llm(prompt):
    loop = asyncio.get_event_loop()

    def sync_call():
        response = requests.post(
            API_URL,
            headers=HEADERS,
            json={
                "model": "openai/gpt-3.5-turbo",
                "messages": [{"role": "user", "content": prompt}]
            }
        )

        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]
        else:
            return f"Error: {response.text}"

    return await loop.run_in_executor(None, sync_call)

# AGENTS

# SEARCH AGENT
async def search_agent(goal, memory):
    print("Search Agent Working...")

    prompt = f"""
    Find relevant data and facts for:
    {goal}
    """

    result = await call_llm(prompt)
    memory["search"] = result
    return result


# ANALYST AGENT
async def analyst_agent(goal, memory):
    print("Analyst Agent Working...")

    prompt = f"""
    Analyze and summarize this information:

    {memory.get('search', '')}
    """

    result = await call_llm(prompt)
    memory["analysis"] = result
    return result


# WRITER AGENT
async def writer_agent(goal, memory):
    print("Writer Agent Working...")

    prompt = f"""
    Write a professional report on:
    {goal}

    Based on:
    {memory.get('analysis', '')}
    """

    result = await call_llm(prompt)
    memory["report"] = result
    return result


#  QA AGENT
async def qa_agent(goal, memory):
    print("QA Agent Working...")

    prompt = f"""
    Review the following report for:
    - Accuracy
    - Clarity
    - Professional tone

    Report:
    {memory.get('report', '')}
    """

    result = await call_llm(prompt)
    memory["final"] = result
    return result


# ORCHESTRATOR (MAIN PIPELINE)
async def orchestrator(goal):
    print("\nStarting AI Research Pipeline...\n")

    memory = {}

    # Step-by-step pipeline
    await search_agent(goal, memory)
    await analyst_agent(goal, memory)
    await writer_agent(goal, memory)
    final = await qa_agent(goal, memory)

    return final


# RUN (COLAB SAFE)
async def main():
    goal = "EV industry trends in India 2025"
    result = await orchestrator(goal)

    print("\nFINAL REPORT:\n")
    print(result)

# Run safely in Colab
await main()


Starting AI Research Pipeline...

Search Agent Working...
Analyst Agent Working...
Writer Agent Working...
QA Agent Working...

FINAL REPORT:

Accuracy:
The report provides accurate information about the current state of the electric vehicle industry in India. It includes specific details about government initiatives, major automakers' entry into the market, and the importance of charging infrastructure. The report also outlines the benefits of electric vehicles accurately. 

Clarity:
The report is well-structured and easy to follow. Each section provides clear and concise information about the topic being discussed, with a logical flow from the introduction to the conclusion. The language used is straightforward and easy to understand, making it accessible to a wide audience.

Professional Tone:
The report maintains a professional tone throughout, using formal language and industry-specific terminology where appropriate. The information presented is objective and supported by evidenc

In [ ]:

# IMPORTS
import asyncio
import requests
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata

API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# LLM CALL FUNCTION
async def call_llm(prompt, temperature=0.7):
    loop = asyncio.get_event_loop()

    def sync_call():
        response = requests.post(
            API_URL,
            headers=HEADERS,
            json={
                "model": "openai/gpt-3.5-turbo",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": temperature
            }
        )

        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]
        else:
            return f"Error: {response.text}"

    return await loop.run_in_executor(None, sync_call)

# AGENTS

# SEARCH AGENT
async def search_agent(goal, memory):
    print("Search Agent Working...")

    prompt = f"""
    Find detailed and relevant data about:
    {goal}
    """

    result = await call_llm(prompt, temperature=0.3)
    memory["search"] = result
    return result


# ANALYST AGENT
async def analyst_agent(goal, memory):
    print("Analyst Agent Working...")

    prompt = f"""
    Analyze and summarize the following data:

    {memory.get('search', '')}
    """

    result = await call_llm(prompt, temperature=0.5)
    memory["analysis"] = result
    return result


# WRITER AGENT
async def writer_agent(goal, memory, feedback=None):
    print("Writer Agent Working...")

    if feedback:
        prompt = f"""
        Improve this report based on feedback:

        Report:
        {memory.get('report', '')}

        Feedback:
        {feedback}
        """
    else:
        prompt = f"""
        Write a professional report on:
        {goal}

        Based on:
        {memory.get('analysis', '')}
        """

    result = await call_llm(prompt, temperature=0.9)
    memory["report"] = result
    return result


# DYNAMIC QA AGENT
async def qa_agent(memory):
    print("QA Agent Checking...")

    prompt = f"""
    Review the following report:

    {memory.get('report', '')}

    If the report is high quality, return:
    PASS

    If not, return:
    FAIL: <specific improvements needed>
    """

    result = await call_llm(prompt, temperature=0.2)
    return result

# ORCHESTRATOR (WITH LOOP)
async def orchestrator(goal):
    print("\nStarting AI Research Pipeline...\n")

    memory = {}

    # Step 1: Search
    await search_agent(goal, memory)

    # Step 2: Analysis
    await analyst_agent(goal, memory)

    # Step 3: Initial Writing
    await writer_agent(goal, memory)

    # Step 4: Dynamic QA Loop
    for i in range(3):   # max 3 retries
        print(f"\nQA Iteration {i+1}")

        qa_result = await qa_agent(memory)

        if "PASS" in qa_result:
            print("Report Approved")
            memory["final"] = memory["report"]
            break
        else:
            print("improving report...")

            # Improve using feedback
            await writer_agent(goal, memory, feedback=qa_result)

    return memory.get("final", memory.get("report"))


# MAIN FUNCTION
async def main():
    goal = "EV industry trends in India 2025"

    result = await orchestrator(goal)

    print("\n📄 FINAL REPORT:\n")
    print(result)


# RUN (COLAB SAFE)
await main()


Starting AI Research Pipeline...

Search Agent Working...
Analyst Agent Working...
Writer Agent Working...

QA Iteration 1
QA Agent Checking...
Report Approved

📄 FINAL REPORT:


Executive Summary:
The electric vehicle (EV) industry in India is poised for significant growth and development by the year 2025. This growth is expected to be driven by various factors such as government incentives, improvements in charging infrastructure, cost reductions, and advancements in technology. The industry is likely to witness an increase in manufacturing, job creation, and collaboration between key stakeholders. With a promising future ahead, the EV industry in India is set to play a crucial role in the country's transition towards sustainable mobility.

Introduction:
The EV industry in India has been steadily growing over the past few years, with the government playing a pivotal role in promoting the adoption of electric vehicles. Initiatives such as the Faster Adoption and Manufacturing of Hybr